# Pruned SAM — 进阶蒸馏训练 (全量 COCO)

**训练策略**: 提示循环蒸馏 (Prompt-in-the-Loop Distillation)
- Teacher: TinySAM (冻结)
- Student: Pruned-M (全参数微调)
- 损失: `L_mask + 0.5*L_feat + 0.1*L_iou`
- 数据: COCO val2017 (5000 图, 36781 实例)

**预计时间**: ~1 小时（T4 GPU）

In [ ]:
# @title 1. 安装依赖
import os
os.chdir('/')  # 避免目录错误

!pip install -q pycocotools tqdm gdown
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 2. 克隆代码
!git clone https://github.com/buaa-zy-2239/pruned-sam.git /content/vista-slam
%cd /content/vista-slam

In [ ]:
# @title 3. 上传模型权重
from google.colab import files
print("请上传 /tmp/vista-slam-data.tar.gz (68MB, 含 pruned_m.pth + tinysam_42.3.pth)")
uploaded = files.upload()
for fn in uploaded:
    !tar xzf "{fn}"
    print(f"✅ 解压 {fn}")

In [ ]:
# @title 4. 下载完整 COCO val2017 (1GB)
import os
if not os.path.exists('eval_data/val2017/000000000139.jpg'):
    !wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip
    !mkdir -p eval_data/val2017
    !unzip -q /tmp/val2017.zip -d eval_data/
    !rm /tmp/val2017.zip
    n = len([f for f in os.listdir('eval_data/val2017') if f.endswith('.jpg')])
    print(f"✅ val2017: {n} 图片")
else:
    print("✅ 已存在")

In [ ]:
# @title 5. 验证文件完整性
import os
checks = [
    ('TinySAM/weights/tinysam_42.3.pth', '教师权重'),
    ('pruned_sam/weights/pruned_m.pth', '学生权重'),
    ('eval_data/annotations/instances_val2017.json', 'COCO标注'),
    ('eval_data/val2017', 'COCO图片'),
]
all_ok = True
for path, name in checks:
    if os.path.exists(path):
        if os.path.isdir(path):
            n = len(os.listdir(path))
            print(f"  ✅ {name}: {n} 文件")
        else:
            print(f"  ✅ {name}: {os.path.getsize(path)/1e6:.0f}MB")
    else:
        print(f"  ❌ {name} 缺失")
        all_ok = False
print(f"\n{'✅ 全部就绪!' if all_ok else '❌ 请检查缺失文件'}")

In [ ]:
# @title 6. 开始进阶蒸馏训练 (全量 COCO)
!python pruned_sam/train_distill_advanced.py --epochs 15 --lr 2e-4 --max_boxes 5

In [ ]:
# @title 7. 下载训练好的模型
from google.colab import files
import glob
for f in sorted(glob.glob('pruned_sam/weights/pruned_m_distill_*.pth')):
    print(f"下载 {os.path.basename(f)} ({os.path.getsize(f)/1e6:.0f}MB)")
    files.download(f)

In [ ]:
# @title 8. (可选) 微调后评估
!python pruned_sam/evaluate_box_miou.py